# From Field Recording to Model-Ready Speech
### *Daga rikodi zuwa corpus* — Deep Learning Indaba, Lagos 2026

Radio stations, oral historians, places of worship and the phones in this room hold thousands of hours of African speech. None of it is a dataset yet.

We have seen a scanned page gain value. Now we do the same with sound. One rough field recording goes in. A small, clean, labelled speech corpus comes out.

The recording is real Hausa: ten words spoken by a Hausa speaker (from Wikimedia Commons, CC0 license). We joined them into one tape the way a field recorder would capture them: background noise, electrical hum, and a speaker who moves closer to and further from the microphone.

Note: cells with a grey title contain setup code. Double-click them to read the code inside.

In [ ]:
#@title Setup — installs and data (double-click to read the code) { display-mode: "form" }
# Installs the audio helpers, downloads the workshop data, and defines the
# listen and plot utilities.
import os, subprocess, sys

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "noisereduce"], capture_output=True)

REPO_URL = "https://github.com/abumafrim/indaba2026-ocr-audio"  # update if you fork
if os.path.exists("../data"):
    ROOT = ".."
elif os.path.exists("data"):
    ROOT = "."
else:
    subprocess.run(["git", "clone", "-q", REPO_URL + ".git"], check=True)
    ROOT = "indaba2026-ocr-audio"
DATA = os.path.join(ROOT, "data")

import librosa
import librosa.display
import matplotlib.pyplot as plt
import noisereduce as nr
import numpy as np
import soundfile as sf
from IPython.display import Audio, display

def listen(y, sr, label=""):
    if label:
        print(label)
    display(Audio(y, rate=sr))

def waveform(*sigs, sr, titles=None, height=2.6):
    fig, axes = plt.subplots(len(sigs), 1, figsize=(14, height * len(sigs)), squeeze=False)
    for ax, y, t in zip(axes[:, 0], sigs, (titles or [""] * len(sigs))):
        librosa.display.waveshow(y, sr=sr, ax=ax)
        ax.set_title(t, fontsize=12)
        ax.set_ylim(-1, 1)
    plt.tight_layout()
    plt.show()

def spectrogram(*sigs, sr, titles=None):
    fig, axes = plt.subplots(1, len(sigs), figsize=(7.5 * len(sigs), 4.5), squeeze=False)
    for ax, y, t in zip(axes[0], sigs, (titles or [""] * len(sigs))):
        S = librosa.power_to_db(librosa.feature.melspectrogram(y=y, sr=sr, n_mels=80), ref=np.max)
        librosa.display.specshow(S, sr=sr, x_axis="time", y_axis="mel", ax=ax)
        ax.set_title(t, fontsize=12)
    plt.tight_layout()
    plt.show()

word_list = open(os.path.join(DATA, "word_list.txt")).read().split()
print("Setup complete.  Expecting", len(word_list), "words:", ", ".join(word_list))

## Part 1 — The asset

One tape from a word-list recording session. Recording word lists is a standard first step when documenting a language. Let us open the file exactly as the recorder saved it.

In [ ]:
tape_path = os.path.join(DATA, "field_recording_raw.wav")
y_raw, sr_raw = librosa.load(tape_path, sr=None, mono=False)

print(f"channels    : {y_raw.shape[0]}  (stereo)")
print(f"sample rate : {sr_raw:,} per second")
print(f"duration    : {y_raw.shape[1] / sr_raw:.1f} s")
print(f"file size   : {os.path.getsize(tape_path) / 1e6:.1f} MB")
listen(y_raw, sr_raw, "The raw tape. Listen for the hiss, the hum, and the uneven voice:")

In [ ]:
waveform(y_raw[0], sr=sr_raw, titles=["The raw tape: 10 words are inside — some loud, some barely above the noise"])

To a model, this file is only numbers: 48,000 of them per second, per channel. Nothing in the file says where the words start, what was said, or who spoke.

## Part 2 — The work

### Step 1 — One channel is enough
The "stereo" from a field recorder is really the same microphone twice. Speech models expect one channel.

In [ ]:
y = librosa.to_mono(y_raw)
print(f"before: {y_raw.shape}  →  after: {y.shape}   (half the numbers, nothing lost)")

### Step 2 — Down to 16,000 samples per second
Speech models usually work at 16 kHz. The higher rate mostly stores frequencies that human speech does not use. Listen to both. Can you hear a difference?

In [ ]:
y16 = librosa.resample(y, orig_sr=sr_raw, target_sr=16_000)
sr = 16_000

listen(y, sr_raw, "At 48,000 samples per second:")
listen(y16, sr, "At 16,000 — one third of the data:")

### Step 3 — Remove the room
The recorder captured the hum and the hiss together with the voice. We give the algorithm one second of "silence" — which is really pure noise — and it subtracts that noise from the whole tape.

In [ ]:
y_clean = nr.reduce_noise(y=y16, sr=sr, y_noise=y16[:sr], stationary=True)

listen(y16, sr, "Before:")
listen(y_clean, sr, "After noise reduction:")

### What the model sees
Models do not hear sound. They look at it, as a *spectrogram*: time runs left to right, pitch runs bottom to top. Compare the two pictures:

In [ ]:
spectrogram(y16, y_clean, sr=sr, titles=["Before: speech buried in noise", "After: ten clear word shapes"])

### Step 4 — Cut the tape into clips
Now that the silence is truly silent, the machine can find the words by itself. We also make every clip equally loud.

In [ ]:
intervals = librosa.effects.split(y_clean, top_db=30)

# join pieces that are closer than 0.3 s — they belong to the same word
segments = []
for s, e in intervals:
    if segments and s - segments[-1][1] < int(0.3 * sr):
        segments[-1][1] = e
    else:
        segments.append([s, e])

print(f"words on the tape: {len(word_list)}   segments found: {len(segments)}\n")

clips = {}
pad = int(0.05 * sr)
for word, (s, e) in zip(word_list, segments):
    clip = y_clean[max(0, s - pad): e + pad]
    clips[word] = clip / (np.max(np.abs(clip)) + 1e-9) * 0.9   # equal loudness
    print(f"  {word:<10} {(e - s) / sr:4.1f} s")

Ten words on the tape, ten clips found. Each clip now has a name. A sound with a label is a training example.

In [ ]:
listen(clips["Bahaushe"], sr, "Bahaushe — a Hausa person:")
listen(clips["Baƙauye"], sr, "Baƙauye — a villager. Note the letter ƙ, the same letter the OCR lost:")

### The result — a small corpus
A model builder never receives "a tape". They receive clips plus a *manifest*: file name, text, duration, speaker, license. Let us finish the job:

In [ ]:
outdir = "hausa_words_corpus"
os.makedirs(outdir, exist_ok=True)

manifest = []
for i, (word, clip) in enumerate(clips.items()):
    fname = f"{i:02d}_{word}.wav"
    sf.write(os.path.join(outdir, fname), clip, sr, subtype="PCM_16")
    manifest.append((fname, word, f"{len(clip) / sr:.2f}", "Gwanki", "CC0"))

print(f"{'file':<18} {'text':<10} {'sec':>5}  speaker  license")
for row in manifest:
    print(f"{row[0]:<18} {row[1]:<10} {row[2]:>5}  {row[3]:<8} {row[4]}")

corpus_mb = sum(os.path.getsize(os.path.join(outdir, f)) for f in os.listdir(outdir)) / 1e6
print(f"\nraw tape: 4.4 MB, unlabelled  →  corpus: {corpus_mb:.2f} MB, labelled and uniform")

## Part 3 — From 23 seconds to 23,000 hours

Everything above was one tape and ten words. A usable speech corpus is thousands of hours, so every step must be automated, checked, and paid for. Between a radio archive and a Hausa speech model sit the people in this room:

- **Media houses** own the recordings — the raw material this pipeline starts from.
- **Speakers and annotators** add the transcripts — the ground truth a machine cannot create.
- **Engineers** scale up the cleaning you just saw.
- **Lawyers** answer the question this notebook has been raising quietly.

Look at the manifest again. *Speaker* and *license* are part of the data. Whose voice is on the tape? Who agreed to what? Who owns the value that cleaning and labelling added? That is where the next session, on IP, copyright and licensing, begins.

> Data is not yet a dataset. The difference is human work, and that work has owners.

---
*The appendix below is for self-study after the workshop.*

---
# Appendix — for self-study
This part is not in the live demo.

## A1. Can speech recognition handle Hausa?
The OCR notebook found no Hausa model in Tesseract. Speech is only a little better. OpenAI's Whisper lists Hausa, but it was trained on very little Hausa audio. Try it and judge for yourself:

```python
# !pip install -q faster-whisper
# from faster_whisper import WhisperModel
# model = WhisperModel("small")
# segs, info = model.transcribe("hausa_words_corpus/00_Bahaushe.wav", language="ha")
# print(list(segs))
```

Speech recognition for low-resource languages improves only when corpora like the one you just built exist at much larger scale — see [Common Voice Hausa](https://commonvoice.mozilla.org/ha), [NaijaVoices](https://naijavoices.com/) (about 1,800 hours of Hausa, Igbo and Yoruba), [Africa Next Voices](https://africanvoices.io/dataset), and [BibleTTS](https://masakhane-io.github.io/bibleTTS/) (high-quality Hausa audio for speech synthesis).

## A2. Better voice activity detection
Our `librosa.effects.split` step uses loudness only. That works in a quiet room but fails in a market or next to a radio. Production pipelines use a trained voice-activity-detection model:

```python
# import torch
# vad, utils = torch.hub.load('snakers4/silero-vad', 'silero_vad')
# speech_ts = utils[0](torch.from_numpy(y_clean), vad, sampling_rate=16_000)
```

## A3. Quality control at corpus scale
With thousands of clips, you check the data with statistics, not with your ears: clip durations, loudness, clipping, wrong sample rates, near-duplicates. Charts like these are what speech-data teams look at every day:

In [ ]:
durs = [len(c) / sr for c in clips.values()]
peaks = [float(np.max(np.abs(c))) for c in clips.values()]

fig, axes = plt.subplots(1, 2, figsize=(13, 3.5))
axes[0].hist(durs, bins=8)
axes[0].set_title("Clip durations (s) — outliers usually mean a split error")
axes[1].hist(peaks, bins=8)
axes[1].set_title("Peak levels — after normalization, all near 0.9")
plt.tight_layout()
plt.show()

## A4. How this demo was made
The tape is 10 real CC0 Hausa recordings by Wikimedia contributor **Gwanki** ([credits](../data/source_words/CREDITS.txt)), joined and degraded by [`scripts/make_audio_assets.py`](../scripts/make_audio_assets.py) to imitate field conditions. Every problem you watched us fix was added on purpose, so the word-level ground truth was known. Real archives have no ground truth until people make it. Transcription and review platforms such as [AfriAnnotate](https://label.afriannotate.org) exist for that work — and they keep track of consent and credit, so that information survives all the way to the model.

## A5. Where to go next
- [Mozilla Common Voice](https://commonvoice.mozilla.org/) — contribute your voice in your language
- [Lacuna Fund](https://lacunafund.org/) — funding for African speech datasets
- [Masakhane](https://www.masakhane.io/) — African NLP community, speech included
- `librosa`, `noisereduce`, `silero-vad`, `faster-whisper` — the exact tools used today